# Experiment 2 — Execution workflow

## Overview

A presentation copy of the original Colab workflow for validation-only model selection. The notebook documents runtime checks, package verification, environment setup, selection, freeze verification and export.

## Reading and setup

- **Read the saved outputs first.** No code was executed or training rerun to prepare this copy.
- **Environment:** Google Colab, with the original `/content` paths and pinned dependencies shown below.
- **Required input:** `asi_robustness_cloud_selection.tar.gz`, containing the `cloud_package` directory and its requirements, runner and selection inputs.
- **Execution order:** original code cells remain in their original sequence. The selection section launches the experiment if executed; the later inspection and export cells depend on existing artifacts.
- **Results:** saved outputs, execution counts and metadata are retained exactly. Section headings describe operations and do not add scientific conclusions.

## Workflow map

Runtime → source package → environment → selection → freeze verification → export.

## 1. Runtime and initial dependency check

Record Python, available memory and LightGBM, then retain the original pinned installation and version check.

In [ ]:
import psutil
import sys
import importlib.metadata

memory = psutil.virtual_memory()

print("Python:", sys.version.split()[0])
print("Total RAM:", round(memory.total / 2**30, 2), "GiB")
print("Available RAM:", round(memory.available / 2**30, 2), "GiB")

try:
    print("LightGBM:", importlib.metadata.version("lightgbm"))
except importlib.metadata.PackageNotFoundError:
    print("LightGBM: Not installed")

Python: 3.13.16
Total RAM: 12.67 GiB
Available RAM: 11.79 GiB
LightGBM: 4.6.0


In [ ]:
%pip install -q "lightgbm==4.7.0"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 30.5 MB/s eta 0:00:00


In [ ]:
try:
    print("LightGBM:", importlib.metadata.version("lightgbm"))
except importlib.metadata.PackageNotFoundError:
    print("LightGBM: Not installed")

LightGBM: 4.7.0


## 2. Upload and verify the source package

The original workflow expects `asi_robustness_cloud_selection.tar.gz` in `/content`. Inspect archive members, extract the package and review the printed checksum result before using the runner.

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving asi_robustness_cloud_selection.tar.gz to asi_robustness_cloud_selection.tar.gz


In [ ]:
from pathlib import Path

p = Path("/content/asi_robustness_cloud_selection.tar.gz")

print("Uploaded:", p.exists())
if p.exists():
    print("Size:", round(p.stat().st_size / 2**20, 2), "MiB")

Uploaded: True
Size: 50.48 MiB


In [ ]:
import tarfile
from pathlib import Path

archive = Path("/content/asi_robustness_cloud_selection.tar.gz")

with tarfile.open(archive, "r:gz") as tar:
    members = tar.getmembers()

    print("Total entries:", len(members))
    print("\nArchive contents:")

    for member in members[:80]:
        print(
            "DIR " if member.isdir() else "FILE",
            member.name,
            f"({member.size:,} bytes)"
        )

    print("\nTotal uncompressed size:",
          round(sum(m.size for m in members if m.isfile()) / 2**20, 2),
          "MiB")

    unsafe = [
        m.name for m in members
        if m.name.startswith("/")
        or ".." in Path(m.name).parts
        or m.issym()
        or m.islnk()
        or m.isdev()
        or m.isfifo()
    ]

    print("Unsafe entries:", unsafe)

Total entries: 25

Archive contents:
FILE cloud_package/CLOUD_PLAN.md (6,956 bytes)
FILE cloud_package/Dockerfile (335 bytes)
FILE cloud_package/environment.json (367 bytes)
FILE cloud_package/pipeline/asi_pipeline/__init__.py (54 bytes)
FILE cloud_package/pipeline/asi_pipeline/checkpoints.py (3,329 bytes)
FILE cloud_package/pipeline/asi_pipeline/dataset.py (5,769 bytes)
FILE cloud_package/pipeline/asi_pipeline/execution.py (15,523 bytes)
FILE cloud_package/pipeline/asi_pipeline/feasibility.py (3,806 bytes)
FILE cloud_package/pipeline/asi_pipeline/metrics.py (5,559 bytes)
FILE cloud_package/pipeline/asi_pipeline/modeling.py (2,805 bytes)
FILE cloud_package/pipeline/asi_pipeline/primary_data.py (15,898 bytes)
FILE cloud_package/pipeline/asi_pipeline/remote_temporal.py (15,470 bytes)
FILE cloud_package/pipeline/asi_pipeline/safety.py (3,894 bytes)
FILE cloud_package/pipeline/asi_pipeline/scientific_run.py (4,499 bytes)
FILE cloud_package/pipeline/asi_pipeline/temporal.py (2,895 bytes)
FI

In [ ]:
import tarfile
import subprocess
from pathlib import Path

archive = Path("/content/asi_robustness_cloud_selection.tar.gz")

with tarfile.open(archive, "r:gz") as tar:
    tar.extractall("/content", filter="data")

result = subprocess.run(
    ["sha256sum", "-c", "SHA256SUMS"],
    cwd="/content/cloud_package",
    text=True,
    capture_output=True
)

print(result.stdout)
print(result.stderr)
print("Checksum verification passed:", result.returncode == 0)

CLOUD_PLAN.md: OK
Dockerfile: OK
environment.json: OK
pipeline/asi_pipeline/__init__.py: OK
pipeline/asi_pipeline/checkpoints.py: OK
pipeline/asi_pipeline/dataset.py: OK
pipeline/asi_pipeline/execution.py: OK
pipeline/asi_pipeline/feasibility.py: OK
pipeline/asi_pipeline/metrics.py: OK
pipeline/asi_pipeline/modeling.py: OK
pipeline/asi_pipeline/primary_data.py: OK
pipeline/asi_pipeline/remote_temporal.py: OK
pipeline/asi_pipeline/safety.py: OK
pipeline/asi_pipeline/scientific_run.py: OK
pipeline/asi_pipeline/temporal.py: OK
pipeline/asi_pipeline/train_cli.py: OK
pipeline/asi_pipeline/windows_monitor.py: OK
protocol_20261020.json: OK
recovery_manifest.json: OK
requirements.txt: OK
run_experiment.py: OK
selection_inputs/train.parquet: OK
selection_inputs/validation.parquet: OK
split_audit.json: OK


Checksum verification passed: True


## 3. Review the execution plan and environment

Inspect the bundled plan, requirements, runner help and Dockerfile. The following cells install the bundled requirements, compare expected versions and check imports.

In [ ]:
from pathlib import Path

root = Path("/content/cloud_package")

print("=== CLOUD PLAN ===")
print((root / "CLOUD_PLAN.md").read_text())

print("\n=== REQUIREMENTS ===")
print((root / "requirements.txt").read_text())

=== CLOUD PLAN ===
# Reproducible cloud continuation — NOT EXECUTED

This is the same post-hoc internal split-sensitivity experiment, not independent external replication. No cloud machine has been provisioned, charged, or trained by this audit.

## Resource and scientific contract

Use a dedicated Linux CPU machine with at least 16 GiB RAM, 4 vCPUs and 20 GiB free disk. No GPU is needed. Run one candidate at a time, with the original two LightGBM threads. Use the container limit below to leave host headroom. If that host fails its memory checks or is OOM-killed, preserve logs and stop; do not reduce rows, features, candidates or boosting budget.

Preserve the existing split files and their SHA-256 hashes; do not reconstruct the split. Use model seed 20260927, split seed 20261020, original M0 R+P+A and M1 R+P+A+O, train-only categorical encoding, all 12 candidates per role, 200 maximum rounds, 30-round early stopping, and lowest-candidate-index tie breaking. No train+validation refit i

In [ ]:
import psutil
import subprocess
from pathlib import Path

root = Path("/content/cloud_package")

mem = psutil.virtual_memory()
print("Total RAM:", round(mem.total / 2**30, 2), "GiB")
print("Available RAM:", round(mem.available / 2**30, 2), "GiB")

result = subprocess.run(
    ["python", "run_experiment.py", "--help"],
    cwd=root,
    capture_output=True,
    text=True
)

print("\n=== RUNNER HELP ===")
print(result.stdout)
print(result.stderr)
print("Exit code:", result.returncode)

print("\n=== DOCKERFILE ===")
print((root / "Dockerfile").read_text())

Total RAM: 12.67 GiB
Available RAM: 11.48 GiB

=== RUNNER HELP ===
usage: run_experiment.py [-h] --out OUT [--test TEST]
                         {select,verify-freeze,evaluate}

positional arguments:
  {select,verify-freeze,evaluate}

options:
  -h, --help            show this help message and exit
  --out OUT
  --test TEST


Exit code: 0

=== DOCKERFILE ===
FROM python:3.12.2-slim-bookworm
RUN apt-get update && apt-get install -y --no-install-recommends libgomp1 && rm -rf /var/lib/apt/lists/*
COPY requirements.txt /tmp/requirements.txt
RUN pip install --no-cache-dir -r /tmp/requirements.txt
ENV PYTHONDONTWRITEBYTECODE=1
WORKDIR /app
ENTRYPOINT ["python", "-u", "/app/run_experiment.py"]



In [ ]:
import sys
import importlib.metadata as md

print("Python:", sys.version)

for package in [
    "lightgbm", "numpy", "pandas", "pyarrow",
    "duckdb", "scipy", "PyYAML", "psutil"
]:
    try:
        print(f"{package}: {md.version(package)}")
    except md.PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")

Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
lightgbm: 4.7.0
numpy: 2.1.3
pandas: 2.2.3
pyarrow: 23.0.1
duckdb: 1.3.2
scipy: 1.16.3
PyYAML: 6.0.3
psutil: 5.9.5


In [ ]:
%pip install -q -r /content/cloud_package/requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.9/18.9 MB 55.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 76.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.2/40.2 MB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 759.5/759.5 kB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 287.5/287.5 kB 21.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tifffile 2026.10.15 requires numpy>=2.1, but you have numpy 2.0.1 which is incompatible

In [ ]:
import sys
import importlib.metadata as md

print("Python:", sys.version.split()[0])

expected = {
    "lightgbm": "4.7.0",
    "numpy": "2.0.1",
    "pandas": "2.2.3",
    "pyarrow": "25.0.1",
    "duckdb": "1.5.5",
    "scipy": "1.15.1",
    "PyYAML": "6.0.2",
    "psutil": "6.1.1",
}

for name, version in expected.items():
    try:
        installed = md.version(name)
        print(f"{name}: {installed} {'OK' if installed == version else 'MISMATCH'}")
    except md.PackageNotFoundError:
        print(f"{name}: NOT INSTALLED")

Python: 3.13.16
lightgbm: 4.7.0 OK
numpy: 2.0.1 OK
pandas: 2.2.3 OK
pyarrow: 25.0.1 OK
duckdb: 1.5.5 OK
scipy: 1.15.1 OK
PyYAML: 6.0.2 OK
psutil: 6.1.1 OK


In [ ]:
import numpy
import pandas
import pyarrow
import duckdb
import scipy
import lightgbm
import psutil
import yaml

print("All experiment libraries imported successfully.")

All experiment libraries imported successfully.


## 4. Execute validation-only selection

This is the training/selection cell. It allocates a fresh `run_colab_NN` directory and records console output and sampled memory under `/content/cloud-logs`. The preserved saved output documents the original execution; it is not a newly generated result.

In [ ]:
import subprocess
import threading
import time
import psutil
import sys
from pathlib import Path

root = Path("/content/cloud_package")
base = Path("/content/cloud-results")
base.mkdir(parents=True, exist_ok=True)

n = 1
while (base / f"run_colab_{n:02d}").exists():
    n += 1

out = base / f"run_colab_{n:02d}"
logs = Path("/content/cloud-logs")
logs.mkdir(parents=True, exist_ok=True)

print("Fresh output directory:", out)

def monitor_memory(stop_event):
    with (logs / "memory.csv").open("w") as f:
        f.write("timestamp,available_gib,used_percent\n")
        while not stop_event.is_set():
            mem = psutil.virtual_memory()
            f.write(
                f"{time.time():.0f},"
                f"{mem.available / 2**30:.3f},"
                f"{mem.percent:.1f}\n"
            )
            f.flush()
            stop_event.wait(5)

stop_event = threading.Event()
monitor = threading.Thread(
    target=monitor_memory,
    args=(stop_event,),
    daemon=True
)

monitor.start()

print("Starting validation-only selection...", flush=True)
print("Results:", out, flush=True)

try:
    with (logs / "selection.log").open("w") as logfile:
        process = subprocess.Popen(
            [
                sys.executable, "-u",
                str(root / "run_experiment.py"),
                "select",
                "--out", str(out),
            ],
            cwd=root,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )

        for line in process.stdout:
            print(line, end="", flush=True)
            logfile.write(line)
            logfile.flush()

        exit_code = process.wait()

finally:
    stop_event.set()
    monitor.join(timeout=10)

print("\nSelection exit code:", exit_code)
print("Memory log:", logs / "memory.csv")
print("Training log:", logs / "selection.log")

Fresh output directory: /content/cloud-results/run_colab_02
Starting validation-only selection...
Results: /content/cloud-results/run_colab_02
{"role": "m0", "candidate_id": 1, "validation_log_mae": 0.8141400482411273, "best_iteration": 21, "parameters": {"objective": "regression_l1", "metric": "l1", "learning_rate": 0.05, "bagging_fraction": 0.8, "bagging_freq": 1, "verbosity": -1, "num_threads": 2, "deterministic": true, "force_col_wise": true, "feature_pre_filter": false, "num_leaves": 15, "min_data_in_leaf": 20, "feature_fraction": 0.8, "seed": 20260927, "bagging_seed": 20260927, "feature_fraction_seed": 20260927, "data_random_seed": 20260927}, "training_seconds": 18.500564120000035, "prediction_seconds": 0.5197178009998424}
{"role": "m0", "candidate_id": 2, "validation_log_mae": 0.7852658401015193, "best_iteration": 141, "parameters": {"objective": "regression_l1", "metric": "l1", "learning_rate": 0.05, "bagging_fraction": 0.8, "bagging_freq": 1, "verbosity": -1, "num_threads": 2,

## 5. Locate results and verify the freeze

These cells inspect result directories and invoke `verify-freeze`. The original paths include both `run` and `run_colab_02`; they are retained exactly and are not automatically linked to the fresh directory chosen above.

In [ ]:
from pathlib import Path

out = Path("/content/cloud-results/run")

print("Exists:", out.exists())
print("Contents:", list(out.iterdir()) if out.is_dir() else "None")

Exists: True
Contents: []


In [ ]:
from pathlib import Path

base = Path("/content/cloud-results")

for name in ["run", "run_colab_01", "run_colab_02"]:
    p = base / name
    print(name, "| exists:", p.exists(),
          "| contents:", [x.name for x in p.iterdir()] if p.is_dir() else "N/A")

run | exists: True | contents: []
run_colab_01 | exists: True | contents: []
run_colab_02 | exists: False | contents: N/A


In [ ]:
import subprocess
import sys

result = subprocess.run(
    [
        sys.executable,
        "/content/cloud_package/run_experiment.py",
        "verify-freeze",
        "--out",
        "/content/cloud-results/run_colab_02",
    ],
    cwd="/content/cloud_package",
    capture_output=True,
    text=True,
)

print(result.stdout)
print(result.stderr)
print("Freeze verification passed:", result.returncode == 0)

Both model selections and all frozen artifact hashes verified.


Freeze verification passed: True


## 6. Export results and execution logs

The original export cells package `cloud-results` and `cloud-logs` separately and download the ZIP archives from Colab.

In [ ]:
from pathlib import Path
import shutil

archive = shutil.make_archive(
    "/content/asi_robustness_colab_selection",
    "zip",
    root_dir="/content",
    base_dir="cloud-results",
)

print("Saved:", archive)
print("Size:", round(Path(archive).stat().st_size / 2**20, 2), "MiB")

Saved: /content/asi_robustness_colab_selection.zip
Size: 0.37 MiB


In [ ]:
from google.colab import files
files.download("/content/asi_robustness_colab_selection.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import shutil
from google.colab import files

shutil.make_archive(
    "/content/asi_robustness_colab_logs",
    "zip",
    root_dir="/content",
    base_dir="cloud-logs",
)

files.download("/content/asi_robustness_colab_logs.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>